# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I frame this as a regression problem because CTR is a continuous target. I will use a Random Forest Regressor because it can capture nonlinear relationships between search-performance features and CTR without assuming a simple linear relationship. The model will be compared with the rule-based baseline using the same evaluation metric and split.

`gsc_clicks` is excluded because CTR is calculated directly from clicks and impressions, so using clicks as a feature would leak information from the target.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The current feature dataset contains only March 2026, so a time-aware split cannot be performed from this subset alone. The modeling dataset must include at least one earlier training period and a later sealed test period before fitting the final model.

In [1]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

In [2]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")

Token loaded successfully!
Connected successfully!


In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [4]:
feature_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

print("Rows:", len(feature_df))
print("Columns:", feature_df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'client_has_gsc', 'client_has_ga4']


In [10]:
months = con.sql("""
SELECT DISTINCT month
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
ORDER BY month
""").df()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


In [11]:
# Load the modeling periods

train_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-4]/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

val_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

test_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 12531047
Validation rows: 4373422
Test rows: 3878937


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [12]:
# Create CTR target
for df in [train_df, val_df, test_df]:
    df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

# Safe features for the model
features = [
    "gsc_impressions",
    "gsc_sum_position",
    "client_has_gsc",
    "client_has_ga4"
]

X_train = train_df[features]
y_train = train_df["ctr"]

X_val = val_df[features]
y_val = val_df["ctr"]

X_test = test_df[features]
y_test = test_df["ctr"]

print("Features:", features)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Features: ['gsc_impressions', 'gsc_sum_position', 'client_has_gsc', 'client_has_ga4']
Training shape: (12531047, 4)
Validation shape: (4373422, 4)
Test shape: (3878937, 4)


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline: predict the average CTR observed in training data
baseline_prediction = np.full(
    len(y_test),
    y_train.mean()
)

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline results")
print("MAE :", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²  :", baseline_r2)

Baseline results
MAE : 0.007251329331887943
RMSE: 0.03693171596579568
R²  : -0.0025399316686165463


In [14]:
from sklearn.ensemble import RandomForestRegressor

# Use a manageable training sample
train_sample = train_df.sample(
    n=500_000,
    random_state=42
)

X_train_sample = train_sample[features]
y_train_sample = train_sample["ctr"]

print("Training sample:", X_train_sample.shape)

Training sample: (500000, 4)


In [15]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_sample, y_train_sample)

print("Model training complete.")

Model training complete.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.